## Plotly 3D Mesh plots

by: Brett Mattas
<br>date: 5/21/2016

### Purpose

The purpose of this notebook is to demonstrate the use of plotly 3D Mesh plots.

Ended up adding Aashto ad-hoc function from CANDE Solutions & Formulations

### Included:

1. Mesh plot using AASHTO ad-hoc function with variable angle.

In [ ]:
import plotly.graph_objects as go
import numpy as np
from buried_structures import Point_3d, Origin
from math import pi, radians, tan
import plotly.io as pio
pio.renderers.default = "notebook"
print("Finished Imports")

Finished Imports


In [2]:
def aashto_30_reduction(z, w0, l0):
    # Return reduction factor of rectangular load assuming
    # 30 degree spread

    w1, l1 = w0 + z * tan(radians(30)), l0 + z * tan(radians(30))

    return (w0 * l0) / (w1 * l1)

def aashto_reduction(z: float, w0: float, l0: float) -> float:
    # Similar, but uses the scaled return
    y = 44 * ((l0 * w0) / 200) ** 0.50 # Equation 8.1-9 in Methods and Formulations

    theta = min(30, z/ (y/2) * 30)

    w1, l1 = w0 + z * tan(radians(theta)), l0 + z * tan(radians(theta))

    return (w0 * l0) / (w1 * l1)

print(aashto_reduction(10, 20, 10))


0.7177087720617876


In [3]:
def find_extreme(iterable, func=max) -> float|None:
    """
    Recursively find the min or max in nested iterables.
    :param iterable: The nested collection (list, tuple, etc.)
    :param func: The built-in min or max function
    """
    flat_elements = []
    
    for item in iterable:
        # Check if the item is a nested iterable (excluding strings if desired)
        if isinstance(item, (list, tuple, set)):
            # Recursive call to drill into the nested structure
            inner_extreme = find_extreme(item, func)
            if inner_extreme is not None:
                flat_elements.append(inner_extreme)
        else:
            # Base case: item is a single value (int, float, etc.)
            flat_elements.append(item)
            
    return func(flat_elements) if flat_elements else None

# Example usage:
nested_data = [1, [5, [10, -2]], 8, [0]]
print(f"Maximum: {find_extreme(nested_data, max)}") # Output: 10
print(f"Minimum: {find_extreme(nested_data, min)}") # Output: -2

Maximum: 10
Minimum: -2


In [4]:
z = np.linspace(0, 48, 50)

l0, w0 = 10, 20 # AASHTO standard load length & width


y = 44 * ((l0 * w0) / 200) ** 0.50 # Equation 8.1-9 in Methods and Formulations

theta = [min(30, iz/ (y/2) * 30) for iz in z]


l = [l0 + iz * tan(radians(itheta)) for iz, itheta in zip(z, theta)]
w = [w0 + iz * tan(radians(itheta)) for iz, itheta in zip(z, theta)]

# r = aashto_30_reduction(z, l0, w0)
r = [aashto_reduction(iz, l0, w0) for iz in z]
# print(r)

rmin, rmax = find_extreme(r, min), find_extreme(r, max)
print(f"{rmin=}, {rmax=}")

rmin=np.float64(0.111149124875471), rmax=np.float64(1.0)


In [5]:
# Setup entire arrays of corners

multipliers = ((1, 1), (-1, 1), (-1, -1), (1, -1))

x, y, zplot, value = [], [], [], []

for iz, il, iw, ir in zip(z, l, w, r):
    for corner in multipliers:
        ix, iy = corner[0] * il/2, corner[1] * iw/2
        # print(f"{ix=}, {iy=}")
        x.append(ix)
        y.append(iy)
        zplot.append(iz)
        value.append(ir)

# print(value)


In [6]:
# x = [5, 5, -5, -5, 6, 6, -6, -6]
# y = [10, -10, 10, -10, 11, -11, 11, -11]
# zplot = [0, 0, 0, 0, 1, 1, 1, 1]
# value = [1, 1, 1, 1, 0.9, 0.9, 0.9, 0.9]

def quad_to_tri(quad: list[int]):
    # Take a list of 4 nodes and return i, j, k lists for two triangles
    i = [quad[0], quad[2]]
    j = [quad[1], quad[3]]
    k = [quad[2], quad[0]]
    return i, j, k



n_layers = len(z)

quad_corners = []

# Add top box
quad_corners = [[0, 1, 2, 3]]

# Add sides
for layer in range(n_layers - 1):

    # Top layer node
    t1 = 4*layer
    t2, t3, t4 = t1+1, t1+2, t1+3

    # Bottom
    b1, b2, b3, b4 = t1+4, t2+4, t3+4, t4+4

    quad_corners.append([b1, b2, t2, t1])
    quad_corners.append([b4, b1, t1, t4])
    quad_corners.append([b3, b4, t4, t3])
    quad_corners.append([b2, b3, t3, t2])
    
# Bottom
b1 = 4*(n_layers-1)
quad_corners.append([b1, b1+1, b1+2, b1+3])
print(quad_corners[-1])

i, j, k = [], [], []
for quad in quad_corners:
    ti, tj, tk = quad_to_tri(quad)
    i.extend(ti)
    j.extend(tj)
    k.extend(tk)



[196, 197, 198, 199]


In [7]:
intensity=np.linspace(0, 1, len(x))
intensity = value


fig = go.Figure(data=[
    go.Mesh3d(
        x=x, y=y, z=zplot, i=i, j=j, k=k,
        intensity=intensity,
        intensitymode='vertex',
        cmin=0, cmax=1,
        colorscale='Turbo',
        showscale=True
    )
])

# Plot points. Useful for degugging.
# fig.add_trace(go.Scatter3d(x=x, y=y, z=zplot, mode='markers', text=[f"Point: {i}" for i in range(len(x))]))

fig.update_layout(title=f"AASHTO 30",
                  width = 800, height=600,
                  scene=dict(
                      xaxis_title="x (in)",
                      yaxis_title = "y (in)",
                      zaxis_title = "z (in)",
                      zaxis=dict(autorange="reversed")
                    )
                  )

fig.show()